# GNN Agricultural Networks — Exploratory Analysis
**Ajinkya Awari** | Initial exploration before model training

This notebook explores the synthetic agricultural graph dataset before running the full training pipeline.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

print('Libraries loaded')

## 1. Generate a small sample graph
Let me first look at a small version (50 nodes) to understand the structure.

In [ ]:
np.random.seed(42)
n = 50
pos = np.random.rand(n, 2)
labels = np.random.choice([0, 1, 2], n, p=[0.5, 0.3, 0.2])

G = nx.Graph()
for i in range(n):
    G.add_node(i, label=labels[i])

delta = 0.3
for i in range(n):
    for j in range(i+1, n):
        if np.linalg.norm(pos[i] - pos[j]) < delta:
            G.add_edge(i, j)

print(f'Nodes: {G.number_of_nodes()}')
print(f'Edges: {G.number_of_edges()}')
print(f'Avg degree: {np.mean([d for _, d in G.degree()]):.2f}')

## 2. Visualise the graph
Colour coding: green = healthy, orange = mild, red = severe

In [ ]:
color_map = {0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}
node_colors = [color_map[labels[i]] for i in range(n)]

plt.figure(figsize=(8, 6))
nx.draw(G, pos=pos, node_color=node_colors, node_size=120,
        edge_color='#cccccc', width=0.5, with_labels=False)
plt.title('Sample Farm Graph (50 nodes)', fontsize=13)
plt.tight_layout()
plt.savefig('sample_graph.png', dpi=100)
plt.show()
print('Saved sample_graph.png')

## 3. Class distribution
Checking class imbalance — important for choosing the right loss function.

In [ ]:
unique, counts = np.unique(labels, return_counts=True)
class_names = ['Healthy', 'Mild', 'Severe']

plt.figure(figsize=(6, 4))
bars = plt.bar(class_names, counts, color=['#2ecc71', '#f39c12', '#e74c3c'])
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(count), ha='center', fontsize=11)
plt.title('Class Distribution', fontsize=13)
plt.ylabel('Number of farms')
plt.tight_layout()
plt.show()

print('Class imbalance detected — will use weighted loss in training')

## 4. Degree distribution
How many neighbours does each farm have? This affects which GNN architecture works best.

In [ ]:
degrees = [d for _, d in G.degree()]

plt.figure(figsize=(6, 4))
plt.hist(degrees, bins=15, color='#457b9d', edgecolor='white')
plt.title('Degree Distribution', fontsize=13)
plt.xlabel('Number of neighbours')
plt.ylabel('Count')
plt.axvline(np.mean(degrees), color='red', linestyle='--',
            label=f'Mean = {np.mean(degrees):.1f}')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Min degree: {min(degrees)}, Max degree: {max(degrees)}, Mean: {np.mean(degrees):.2f}')
print('Relatively uniform degree distribution — this explains why GAT attention did not help')

## 5. Observations before training

- Class imbalance exists (more healthy farms) → need weighted loss ✓
- Degree distribution is fairly uniform → GAT attention may not add much
- Disease clusters are spatially correlated → GNN neighbourhood aggregation should help
- Will run full training pipeline via `python run_all.py`